# LocalStoryChat: Qwen3.5-9B Writer QLoRA on Kaggle\n\nThis notebook runs the first writer-training experiment on a Kaggle T4×2 notebook. It clones the `generic-story-engine-rebuild` branch, rebuilds the Gold dataset locally, checks the GPU, and launches the T4-compatible Axolotl configuration.

In [ ]:
import os, subprocess, sys\nprint(sys.version)\nsubprocess.run(['nvidia-smi'], check=False)\n

## 1. Get the project\n\nKaggle notebook Settings should have **Accelerator = T4x2** and Internet enabled.

In [ ]:
import os, subprocess\nif not os.path.exists('/kaggle/working/LocalStoryChat'):\n    subprocess.run(['git','clone','--branch','generic-story-engine-rebuild','--single-branch','https://github.com/tony21675/LocalStoryChat.git','/kaggle/working/LocalStoryChat'], check=True)\nos.chdir('/kaggle/working/LocalStoryChat')\nprint(os.getcwd())\n

## 2. Install the current training stack\n\nQwen3.5 is currently supported by Axolotl. Cut Cross Entropy can reduce training VRAM use.

In [ ]:
%pip install -q --no-build-isolation \\n   "axolotl>=0.16.1" \\n   "cut-cross-entropy[transformers] @ git+https://github.com/axolotl-ai-cloud/ml-cross-entropy.git@3574df5"\n

If Kaggle reports that packages changed underneath the running kernel, use **Run → Restart Session**, then continue with the next cells. A restart is sometimes needed after large ML package installs.

In [ ]:
import os, subprocess\nos.chdir('/kaggle/working/LocalStoryChat')\nsubprocess.run(['python','training/prepare_gold_dataset.py'], check=True)\n

## 3. Confirm the training inputs\n\nThe notebook rebuilds `gold_dataset.jsonl` from the 48 Gold examples, so the generated dataset does not need to be committed to GitHub.

In [ ]:
from pathlib import Path\ngold = sorted(Path('training/gold').glob('*.json'))\ndataset = Path('training/gold_dataset.jsonl')\nprint('Gold examples:', len(gold))\nprint('Dataset exists:', dataset.exists())\nif dataset.exists():\n    print('Dataset size:', dataset.stat().st_size, 'bytes')\n

## 4. Train\n\nThe first run is deliberately conservative: QLoRA 4-bit, sequence length 2048, micro-batch 1, gradient accumulation 8, three epochs, FP16, and no sample packing.

In [ ]:
import os, subprocess\nos.chdir('/kaggle/working/LocalStoryChat')\nsubprocess.run(['axolotl','train','training/qwen3_5_9b_kaggle_t4.yaml'], check=True)\n

## 5. Locate the adapter\n\nThe trained LoRA adapter should be under `training/outputs/qwen3_5_9b_writer_kaggle/`. Keep the adapter before doing any merge or conversion.

In [ ]:
from pathlib import Path\nout = Path('training/outputs/qwen3_5_9b_writer_kaggle')\nprint('Output exists:', out.exists())\nif out.exists():\n    for p in sorted(out.rglob('*')):\n        if p.is_file():\n            print(p)\n

## 6. Save the adapter\n\nDownload the output directory from the Kaggle file browser, or zip it first with the next cell.

In [ ]:
import shutil\nshutil.make_archive('/kaggle/working/qwen3_5_9b_writer_kaggle_adapter', 'zip', 'training/outputs/qwen3_5_9b_writer_kaggle')\nprint('/kaggle/working/qwen3_5_9b_writer_kaggle_adapter.zip')\n